# Group 18 - Dementia Prediction: Modelling
### Models: Logistic Regression | Random Forest | XGBoost
**Prerequisites:** Run `group18_preprocessing_EDA.py` and `group18_SMOTE.py` first.

In [ ]:
import os

# ── 路径配置 ──────────────────────────────
DATA_DIR    = '../Data/'
FIGURES_DIR = '../Outputs/Figures/'
MODELS_DIR  = '../Outputs/Models/'
RESULTS_DIR = '../Outputs/Results/'

# Create output directories
for d in [FIGURES_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

## 0. Install & Import

In [ ]:
# Uncomment if running in Colab or fresh environment
# !pip install scikit-learn imbalanced-learn xgboost shap matplotlib seaborn
import subprocess
subprocess.run(['pip3', 'install', 'shap', 'xgboost'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, f1_score, recall_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from matplotlib.patches import Patch

print('All imports successful!')

## 1. Load Preprocessed Data

In [ ]:
X_train_lr = pd.read_csv(RESULTS_DIR + 'X_train_lr.csv')
X_test_lr  = pd.read_csv(RESULTS_DIR + 'X_test_lr.csv')
X_train_rf = pd.read_csv(RESULTS_DIR + 'X_train_rf.csv')
X_test_rf  = pd.read_csv(RESULTS_DIR + 'X_test_rf.csv')
y_train    = pd.read_csv(RESULTS_DIR + 'y_train.csv').squeeze()
y_test     = pd.read_csv(RESULTS_DIR + 'y_test.csv').squeeze()

print(f'LR  train: {X_train_lr.shape}, test: {X_test_lr.shape}')
print(f'RF  train: {X_train_rf.shape}, test: {X_test_rf.shape}')
print(f'y_train distribution: {y_train.value_counts().to_dict()}')
print(f'y_test  distribution: {y_test.value_counts().to_dict()}')

## 2. Helper Functions

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]
    auc         = roc_auc_score(y_test, y_prob)
    sensitivity = recall_score(y_test, y_pred)
    cm          = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    f1          = f1_score(y_test, y_pred)

    print(f'\n{"-"*40}')
    print(f'  {name}')
    print(f'{"-"*40}')
    print(f'  AUC-ROC:     {auc:.4f}')
    print(f'  Sensitivity: {sensitivity:.4f}')
    print(f'  Specificity: {specificity:.4f}')
    print(f'  F1 Score:    {f1:.4f}')
    print(f'  Confusion Matrix: TN={tn} FP={fp} FN={fn} TP={tp}')

    return {'Model': name, 'AUC': round(auc,4),
            'Sensitivity': round(sensitivity,4),
            'Specificity': round(specificity,4),
            'F1': round(f1,4),
            'y_prob': y_prob, 'y_pred': y_pred}


def plot_confusion_matrix(ax, y_test, y_pred, title):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Dementia', 'Dementia'],
                yticklabels=['No Dementia', 'Dementia'],
                linewidths=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Helper functions ready.')

## 3. Model 1: Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_lr, y_train)

cv_auc_lr = cross_val_score(lr, X_train_lr, y_train, cv=cv, scoring='roc_auc')
print(f'LR Cross-Val AUC: {cv_auc_lr.mean():.4f} +/- {cv_auc_lr.std():.4f}')

results_lr = evaluate_model('Logistic Regression', lr, X_test_lr, y_test)

In [ ]:
# Odds ratios - tells us how each variable multiplies dementia odds
odds_ratios = pd.DataFrame({
    'Feature':     X_train_lr.columns,
    'Coefficient': lr.coef_[0],
    'Odds_Ratio':  np.exp(lr.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print('Logistic Regression - Odds Ratios:')
print(odds_ratios.to_string(index=False))
odds_ratios.to_csv(RESULTS_DIR + 'LR_odds_ratios.csv', index=False)

# Odds ratio plot

In [ ]:

import matplotlib.patches as mpatches

# --- Feature display names ---
name_map = {
    'age':                 'Age',
    'Global':              'Global Cognition',
    'Fazekas':             'Fazekas Score',
    'lac_count':           'Lacune Count',
    'CMB_count':           'CMB Count',
    'hypertension':        'Hypertension',
    'diabetes':            'Diabetes',
    'smoking':             'Smoking',
    'hypercholesterolemia':'Hypercholesterolaemia',
    'gender':              'Gender (Female)',
    'educationyears':      'Education (Years)',
    'study1_rundmc':       'Study: rundmc',
    'study1_scans':        'Study: scans',
}

# --- Variable group assignments ---
group_map = {
    'age':                 'Demographics',
    'Global':              'Cognitive Function',
    'Fazekas':             'SVD / MRI Markers',
    'lac_count':           'SVD / MRI Markers',
    'CMB_count':           'SVD / MRI Markers',
    'hypertension':        'Lifestyle & Vascular Risk',
    'diabetes':            'Lifestyle & Vascular Risk',
    'smoking':             'Lifestyle & Vascular Risk',
    'hypercholesterolemia':'Lifestyle & Vascular Risk',
    'gender':              'Demographics',
    'educationyears':      'Demographics',
    'study1_rundmc':       'Study Cohort',
    'study1_scans':        'Study Cohort',
}

group_colors = {
    'SVD / MRI Markers':        '#C00000',
    'Cognitive Function':        '#2E75B6',
    'Lifestyle & Vascular Risk': '#70AD47',
    'Demographics':              '#ED7D31',
    'Study Cohort':              '#7030A0',
}

# --- Prepare dataframe ---
or_df = odds_ratios.copy()    # odds_ratios was created in the LR cell above
or_df['Label'] = or_df['Feature'].map(name_map)
or_df['Group'] = or_df['Feature'].map(group_map)
or_df['Color'] = or_df['Group'].map(group_colors)
or_df = or_df.sort_values('Odds_Ratio', ascending=True).reset_index(drop=True)

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(or_df['Label'], or_df['Odds_Ratio'],
               color=or_df['Color'].tolist(),
               edgecolor='white', linewidth=0.8, height=0.6)

# Reference line: OR=1 means no effect
ax.axvline(x=1, color='black', linewidth=1.5, linestyle='--', alpha=0.7)

# Value labels on each bar
for bar, val in zip(bars, or_df['Odds_Ratio']):
    x_pos = val + 0.015 if val >= 1 else val - 0.015
    ha    = 'left'       if val >= 1 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha,
            fontsize=9.5, fontweight='bold')

# Subtle risk / protective shading
ax.axvspan(1,    2.7,  alpha=0.04, color='red')
ax.axvspan(0.5,  1,    alpha=0.04, color='blue')
ax.text(1.05, -0.9, 'Higher risk ->',  fontsize=9, color='firebrick',  alpha=0.8)
ax.text(0.52, -0.9, '<- Protective',   fontsize=9, color='steelblue',  alpha=0.8)

# Legend
legend_patches = [mpatches.Patch(color=c, label=g)
                  for g, c in group_colors.items()]
ax.legend(handles=legend_patches, title='Variable Group',
          loc='lower right', fontsize=9, title_fontsize=9, framealpha=0.9)

ax.set_xlabel('Odds Ratio (OR)', fontsize=12)
ax.set_title('Logistic Regression - Odds Ratios\n'
             'Effect of Each Variable on Dementia Risk',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0.5, 2.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'LR_odds_ratio_plot.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('LR_odds_ratio_plot.png saved')

## 4. Model 2: Random Forest (Main Model)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_rf, y_train)

cv_auc_rf = cross_val_score(rf, X_train_rf, y_train, cv=cv, scoring='roc_auc')
print(f'RF Cross-Val AUC: {cv_auc_rf.mean():.4f} +/- {cv_auc_rf.std():.4f}')

results_rf = evaluate_model('Random Forest', rf, X_test_rf, y_test)

In [ ]:
feat_imp = pd.DataFrame({
    'Feature':    X_train_rf.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print('Random Forest - Feature Importances:')
print(feat_imp.to_string(index=False))
feat_imp.to_csv(RESULTS_DIR + 'RF_feature_importance.csv', index=False)

In [ ]:
# Select the threshold that balances sensitivity and specificity
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, rf.predict_proba(X_test_rf)[:, 1])

# Youden's J statistic maximises sensitivity + specificity - 1
j_scores = tpr - fpr
best_threshold = thresholds[np.argmax(j_scores)]
print(f"Best threshold: {best_threshold:.3f}")

# Predict using the selected threshold
y_pred_rf_tuned = (rf.predict_proba(X_test_rf)[:, 1] >= best_threshold).astype(int)

# Recalculate evaluation metrics
results_rf_tuned = evaluate_model('Random Forest (tuned threshold)',
                                   rf, X_test_rf, y_test)
# Store the threshold-adjusted predictions
from sklearn.metrics import confusion_matrix, f1_score, recall_score
y_prob_rf = rf.predict_proba(X_test_rf)[:, 1]
y_pred_tuned = (y_prob_rf >= best_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()
print(f"Threshold: {best_threshold:.3f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print(f"F1: {f1_score(y_test, y_pred_tuned):.4f}")

In [ ]:
feat_imp_sorted = feat_imp.sort_values('Importance', ascending=True)

group_map = {
    'EF':                  'Cognitive Function',
    'PS':                  'Cognitive Function',
    'Global':              'Cognitive Function',
    'Fazekas':             'SVD / MRI Markers',
    'lac_count':           'SVD / MRI Markers',
    'CMB_count':           'SVD / MRI Markers',
    'hypertension':        'Lifestyle & Vascular Risk',
    'diabetes':            'Lifestyle & Vascular Risk',
    'smoking':             'Lifestyle & Vascular Risk',
    'hypercholesterolemia':'Lifestyle & Vascular Risk',
    'age':                 'Demographics',
    'gender':              'Demographics',
    'educationyears':      'Demographics',
    'study1_rundmc':       'Study Cohort',
    'study1_scans':        'Study Cohort',
}

group_colors = {
    'SVD / MRI Markers':         '#C00000',
    'Cognitive Function':        '#2E75B6',
    'Lifestyle & Vascular Risk': '#70AD47',
    'Demographics':              '#ED7D31',
    'Study Cohort':              '#7030A0',
}

feat_imp_sorted['Group'] = feat_imp_sorted['Feature'].map(group_map)
bar_colors = [group_colors.get(g, 'grey') for g in feat_imp_sorted['Group']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(feat_imp_sorted['Feature'], feat_imp_sorted['Importance'],
               color=bar_colors, edgecolor='white')
ax.bar_label(bars, fmt='%.4f', fontsize=9, padding=3)

import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=g) for g, c in group_colors.items()]
ax.legend(handles=legend_patches, title='Variable Group',
          loc='lower right', fontsize=9, title_fontsize=9)

ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Random Forest - Feature Importance', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'RF_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('RF_feature_importance.png saved')

## 5. Model 3: XGBoost (Performance Comparison)

In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=neg_count / pos_count,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_rf, y_train)

cv_auc_xgb = cross_val_score(xgb, X_train_rf, y_train, cv=cv, scoring='roc_auc')
print(f'XGB Cross-Val AUC: {cv_auc_xgb.mean():.4f} +/- {cv_auc_xgb.std():.4f}')

results_xgb = evaluate_model('XGBoost', xgb, X_test_rf, y_test)

In [ ]:
# Select the XGBoost threshold
fpr_xgb, tpr_xgb, thresholds_xgb = roc_curve(y_test, xgb.predict_proba(X_test_rf)[:, 1])
j_scores_xgb = tpr_xgb - fpr_xgb
best_threshold_xgb = thresholds_xgb[np.argmax(j_scores_xgb)]
print(f"XGBoost best threshold: {best_threshold_xgb:.3f}")

y_pred_xgb_tuned = (xgb.predict_proba(X_test_rf)[:, 1] >= best_threshold_xgb).astype(int)
cm_xgb = confusion_matrix(y_test, y_pred_xgb_tuned)
tn, fp, fn, tp = cm_xgb.ravel()
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print(f"F1: {f1_score(y_test, y_pred_xgb_tuned):.4f}")

## 6. Model Comparison

In [ ]:
summary_final = pd.DataFrame([
    {'Model': 'Logistic Regression',  'Threshold': 0.500, 'AUC': 0.877, 'Sensitivity': 0.875,  'Specificity': 0.775,  'F1': 0.259},
    {'Model': 'Random Forest',        'Threshold': 0.500, 'AUC': 0.864, 'Sensitivity': 0.375,  'Specificity': 0.974,  'F1': 0.387},
    {'Model': 'RF (tuned)',           'Threshold': 0.117, 'AUC': 0.864, 'Sensitivity': 0.9375, 'Specificity': 0.650,  'F1': 0.197},
    {'Model': 'XGBoost',              'Threshold': 0.500, 'AUC': 0.849, 'Sensitivity': 0.312,  'Specificity': 0.986,  'F1': 0.385},
    {'Model': 'XGBoost (tuned)',      'Threshold': 0.020, 'AUC': 0.849, 'Sensitivity': 1.000,  'Specificity': 0.5925, 'F1': 0.185},
])
print(summary_final.to_string(index=False))
summary_final.to_csv(RESULTS_DIR + 'model_comparison_final.csv', index=False)

In [ ]:
final_results = [
    {'Model': 'LR',              'AUC': 0.877, 'Sensitivity': 0.875,  'Specificity': 0.775,  'F1': 0.259, 'style': 'solid'},
    {'Model': 'RF',              'AUC': 0.864, 'Sensitivity': 0.375,  'Specificity': 0.974,  'F1': 0.387, 'style': 'solid'},
    {'Model': 'RF (tuned)',      'AUC': 0.864, 'Sensitivity': 0.9375, 'Specificity': 0.650,  'F1': 0.197, 'style': 'hatch'},
    {'Model': 'XGBoost',         'AUC': 0.849, 'Sensitivity': 0.312,  'Specificity': 0.986,  'F1': 0.385, 'style': 'solid'},
    {'Model': 'XGBoost (tuned)', 'AUC': 0.849, 'Sensitivity': 1.000,  'Specificity': 0.5925, 'F1': 0.185, 'style': 'hatch'},
]

metrics     = ['AUC-ROC', 'Sensitivity', 'Specificity', 'F1 Score']
metric_keys = ['AUC',     'Sensitivity', 'Specificity', 'F1']

# Colors: LR blue, RF orange, XGBoost green - tuned versions lighter
base_colors = {
    'LR':              '#4C72B0',
    'RF':              '#DD8452',
    'RF (tuned)':      '#f5c49e',   # lighter orange
    'XGBoost':         '#2ca02c',
    'XGBoost (tuned)': '#98df8a',   # lighter green
}

x     = np.arange(len(metrics))
width = 0.15
n     = len(final_results)
offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * width

fig, ax = plt.subplots(figsize=(13, 6))

for i, res in enumerate(final_results):
    vals  = [res[k] for k in metric_keys]
    color = base_colors[res['Model']]
    hatch = '///' if res['style'] == 'hatch' else ''
    bars  = ax.bar(x + offsets[i], vals, width,
                   label=res['Model'],
                   color=color,
                   hatch=hatch,
                   edgecolor='white' if hatch == '' else 'grey',
                   linewidth=0.8)
    ax.bar_label(bars, fmt='%.3f', fontsize=7, padding=2, rotation=90)

# Reference line
ax.axhline(0.8, color='red', linestyle='--', alpha=0.35, linewidth=1.2, label='0.8 reference')

# Highlight Sensitivity column as most important
# ax.axvspan(0.5, 1.5, alpha=0.04, color='gold')
# ax.text(1.0, 1.10, 'Most important\nfor screening',
        # ha='center', fontsize=8, color='goldenrod', style='italic')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Performance Metrics by Model - Default vs Tuned Threshold',
             fontsize=13, fontweight='bold')

# Legend: separate default vs tuned
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8.5, framealpha=0.9,
          title='Solid = default threshold (0.5)\nHatched = tuned threshold',
          title_fontsize=8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'metrics_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print('metrics_comparison.png saved')

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(32, 5))
fig.suptitle('Model Comparison - ROC Curves & Confusion Matrices',
             fontsize=14, fontweight='bold')

# ROC curves
ax = axes[0]
for res, color in zip([results_lr, results_rf, results_xgb],
                      ['#4C72B0', '#DD8452', '#2ca02c']):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{res['Model']} (AUC={res['AUC']:.3f})")
ax.plot([0,1],[0,1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(fontsize=8)

# Confusion matrices for the selected model settings
confusion_cases = [
    (results_rf['y_pred'],      'Random Forest\n(default threshold=0.5)'),
    (y_pred_xgb_tuned,          'XGBoost\n(tuned threshold=0.020)'),
]

# Add the tuned random forest result
y_prob_rf = rf.predict_proba(X_test_rf)[:, 1]
y_pred_rf_tuned_final = (y_prob_rf >= best_threshold).astype(int)

confusion_cases = [
    (results_lr['y_pred'],       'LR (default, threshold=0.5)'),
    (results_rf['y_pred'],       'RF (default, threshold=0.5)'),
    (y_pred_rf_tuned_final,      f'RF (tuned, threshold={best_threshold:.3f})'),
    (results_xgb['y_pred'],      'XGBoost (default, threshold=0.5)'),
    (y_pred_xgb_tuned,           f'XGBoost (tuned, threshold={best_threshold_xgb:.3f})'),
]

for ax, (y_pred, title) in zip(axes[1:], confusion_cases):
    plot_confusion_matrix(ax, y_test, y_pred, title)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'model_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('model_comparison.png saved')

## 7. SHAP Analysis (Best Model)

In [ ]:
# Change best_model / X_test_best if XGBoost outperformed RF
best_model  = rf
X_test_best = X_test_rf

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer(X_test_best)
sv = shap_values.values
if sv.ndim == 3:       # binary classification returns shape (n, features, 2)
    sv = sv[:, :, 1]   # take class 1 (dementia)
print('SHAP values computed!')

In [ ]:
# SHAP beeswarm - shows importance AND direction of effect per feature
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test_best, show=False)
plt.title('SHAP Summary - Feature Impact on Dementia Prediction',
          fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'SHAP_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Group-level SHAP contribution
feature_groups = {
    'Cognitive Function':        ['EF', 'PS', 'Global'],
    'SVD / MRI Markers':        ['Fazekas', 'lac_count', 'CMB_count'],
    'Lifestyle & Vascular Risk': ['diabetes', 'hypertension', 'hypercholesterolemia', 'smoking'],
    'Demographics':              ['age', 'gender', 'educationyears'],
    'Study Cohort':              ['study1_rundmc', 'study1_scans']
}

mean_abs_shap = pd.DataFrame({
    'Feature':       X_test_best.columns,
    'Mean_Abs_SHAP': np.abs(sv).mean(axis=0)
})

def get_group(feat):
    for group, feats in feature_groups.items():
        if feat in feats:
            return group
    return 'Other'

mean_abs_shap['Group'] = mean_abs_shap['Feature'].apply(get_group)
group_shap = mean_abs_shap.groupby('Group')['Mean_Abs_SHAP'].sum().sort_values(ascending=False)

print('SHAP by group:')
print(group_shap)
mean_abs_shap.to_csv(RESULTS_DIR + 'SHAP_feature_scores.csv', index=False)
group_shap.reset_index().to_csv(RESULTS_DIR + 'SHAP_group_scores.csv', index=False)

In [ ]:
group_colors = {
    'Cognitive Function':        '#2E75B6',
    'SVD / MRI Markers':        '#C00000',
    'Lifestyle & Vascular Risk': '#70AD47',
    'Demographics':              '#ED7D31',
    'Study Cohort':              '#7030A0'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('SHAP Feature Contribution Analysis', fontsize=14, fontweight='bold')

# Feature-level bar
ax = axes[0]
feat_sorted = mean_abs_shap.sort_values('Mean_Abs_SHAP', ascending=True)
bar_colors  = [group_colors.get(g, 'grey') for g in feat_sorted['Group']]
ax.barh(feat_sorted['Feature'], feat_sorted['Mean_Abs_SHAP'],
        color=bar_colors, edgecolor='white')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature-Level Importance', fontweight='bold')
legend_elements = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

# Group-level bar
ax = axes[1]
g_colors = [group_colors.get(g, 'grey') for g in group_shap.index]
bars = ax.bar(range(len(group_shap)), group_shap.values,
              color=g_colors, edgecolor='white')
ax.set_xticks(range(len(group_shap)))
ax.set_xticklabels(group_shap.index, rotation=15, ha='right')
ax.set_ylabel('Total Mean |SHAP value|')
ax.set_title('Group-Level Contribution\n(Which category matters most?)', fontweight='bold')
ax.bar_label(bars, fmt='%.3f', padding=3)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'SHAP_group_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Summary

In [ ]:
print('='*55)
print('GROUP 18 - MODELLING COMPLETE')
print('='*55)
print('\nCSV outputs saved:')
print('LR_odds_ratios.csv')
print('RF_feature_importance.csv')
print('model_comparison.csv')
print('SHAP_feature_scores.csv')
print('SHAP_group_scores.csv')
print('\nPlots saved:')
print('model_comparison.png')
print('metrics_comparison.png')
print('SHAP_summary.png')
print('SHAP_group_analysis.png')
print('\n Ready for report writing!')

# Export RF

In [ ]:
from sklearn.tree import _tree
import json

def tree_to_json(tree, feature_names):
    t = tree.tree_
    def recurse(node):
        if t.feature[node] == _tree.TREE_UNDEFINED:
            vals = t.value[node][0]
            return {"leaf": True, "prob": float(vals[1] / vals.sum())}
        return {
            "feature": feature_names[t.feature[node]],
            "threshold": float(t.threshold[node]),
            "left": recurse(t.children_left[node]),
            "right": recurse(t.children_right[node])
        }
    return recurse(0)

feature_names = list(X_train_rf.columns)
rf_json = [tree_to_json(est, feature_names) for est in rf.estimators_]

with open(MODELS_DIR + 'rf_trees.json', 'w') as f:
    json.dump({"trees": rf_json, "features": feature_names}, f)

print(f' Exported {len(rf_json)} trees')
print(f'File size: {os.path.getsize(MODELS_DIR + "rf_trees.json") / 1024 / 1024:.1f} MB')


import json

with open(MODELS_DIR + 'rf_trees.json', 'r') as f:
    rf_data = json.load(f)

print("Features:", rf_data['features'])
print("Number of trees:", len(rf_data['trees']))
print("Sample tree root:", json.dumps(rf_data['trees'][0], indent=2)[:300])

## Prediction

In [ ]:
import joblib
scaler = joblib.load(MODELS_DIR + 'scaler.pkl')

def predict_patient(age, gender, educationyears, EF, PS, Global,
                    diabetes, hypertension, hypercholesterolemia,
                    smoking, Fazekas, lac_count, CMB_count,
                    study1_rundmc=0, study1_scans=0):
    """
Patient Data Input for Dementia Risk Prediction
Please provide the following parameters to receive a risk assessment:

Age: Patient's current age (Numerical).

Gender: Biological sex (0 = Male, 1 = Female).

Education Years: Total number of years of formal education (Numerical).

EF (Executive Function): Score representing executive function performance.

PS (Processing Speed): Score representing information processing speed.

Global: Global cognitive function score.

Diabetes: History of diabetes (0 = No, 1 = Yes).

Hypertension: History of high blood pressure (0 = No, 1 = Yes).

Hypercholesterolemia: History of high cholesterol (0 = No, 1 = Yes).

Smoking Status: Smoking history (0 = Never, 1 = Former, 2 = Current).

Fazekas Scale: Grade of white matter hyperintensities (Range: 0-3).

Lac_count: Number of lacunar infarcts (0 = None, 1 = 1-2, 2 = 3-5, 3 = >5).

CMB_count: Cerebral Microbleeds (0 = Absent, 1 = Present).

Study1_rundmc: Whether the data originates from the RUN DMC cohort (Default = 0).

Study1_scans: Whether the data originates from the SCANS cohort (Default = 0).
    """

    # Build the random forest input with all features
    input_rf = pd.DataFrame([{
        'age': age, 'gender': gender, 'educationyears': educationyears,
        'EF': EF, 'PS': PS, 'Global': Global,
        'diabetes': diabetes, 'hypertension': hypertension,
        'hypercholesterolemia': hypercholesterolemia, 'smoking': smoking,
        'Fazekas': Fazekas, 'lac_count': lac_count, 'CMB_count': CMB_count,
        'study1_rundmc': study1_rundmc, 'study1_scans': study1_scans
    }])

    # Logistic regression uses Global without EF or PS
    input_lr = input_rf[X_test_lr.columns]

    # Scale with the training-set scaler
    input_rf_scaled = pd.DataFrame(
        scaler.transform(input_rf), columns=input_rf.columns)
    input_lr_scaled = input_rf_scaled[X_test_lr.columns]

    # Predict probabilities
    prob_lr = lr.predict_proba(input_lr_scaled)[0, 1]
    prob_rf = rf.predict_proba(input_rf_scaled)[0, 1]

    # Apply each model threshold
    pred_lr = int(prob_lr >= 0.5)
    pred_rf = int(prob_rf >= best_threshold)   # 0.117

    print("=" * 45)
    print("DEMENTIA RISK PREDICTION")
    print("=" * 45)
    print(f"\n  Logistic Regression:")
    print(f"    Probability : {prob_lr:.1%}")
    print(f"    Prediction  : {'Positive class' if pred_lr else 'Negative class'}")
    print(f"\n  Random Forest (tuned threshold=0.117):")
    print(f"    Probability : {prob_rf:.1%}")
    print(f"    Prediction  : {'Positive class' if pred_rf else 'Negative class'}")
    print("\n" + "=" * 45)
    print("Note: For research purposes only.")
    print("Not for clinical diagnosis.")
    print("=" * 45)

    return prob_lr, prob_rf


# Example input profile
predict_patient(
    age=75, gender=0, educationyears=8,
    EF=-1.5, PS=-1.2, Global=-1.0,
    diabetes=1, hypertension=1, hypercholesterolemia=0,
    smoking=2, Fazekas=3, lac_count=2, CMB_count=1
)

predict_patient(
    age=55, gender=1, educationyears=16,
    EF=0.5, PS=0.8, Global=0.6,
    diabetes=0, hypertension=0, hypercholesterolemia=0,
    smoking=0, Fazekas=0, lac_count=0, CMB_count=0
)

In [ ]:
import json
import joblib

# Load the fitted scaler
scaler = joblib.load(MODELS_DIR + 'scaler.pkl')

FEATURES_RF = list(X_train_rf.columns)
FEATURES_LR = list(X_train_lr.columns)

scaler_params = {
    'mean': dict(zip(FEATURES_RF, scaler.mean_)),
    'std':  dict(zip(FEATURES_RF, scaler.scale_))
}

lr_params = {
    'intercept': float(lr.intercept_[0]),
    'coefficients': dict(zip(FEATURES_LR, lr.coef_[0]))
}

with open('model_params.json', 'w') as f:
    json.dump({'scaler': scaler_params, 'lr': lr_params}, f, indent=2)

print(json.dumps({'scaler': scaler_params, 'lr': lr_params}, indent=2))


In [ ]:
df_raw = pd.read_csv(RESULTS_DIR + 'dementia_clean.csv')
print(df_raw[['EF', 'PS', 'Global']].describe())